# OCR Post-processing & Error Correction

Mục tiêu: Áp dụng các phương pháp post-processing để sửa lỗi đầu ra tự động từ mô hình OCR (Experiment B), nhằm gia tăng tỉ lệ Exact Match.

Hai phương pháp cần thử nghiệm:
1. **Lexicon/SymSpell**: Dựa trên từ điển và edit distance (mô phỏng một phần idea từ Unsupervised Error Correction paper)
2. **BARTpho (SOTA LLM)**: Sử dụng mô hình xử lý ngôn ngữ học tự nhiên mạnh mẽ cho tiếng Việt.

In [ ]:
!pip install -q transformers sentencepiece symspellpy pyvi peft accelerate jiwer

In [ ]:
import pandas as pd
import numpy as np
import re
import torch
from pathlib import Path
from tqdm import tqdm
from jiwer import cer, wer
import warnings
warnings.filterwarnings("ignore")

## 1. Load Experiment B Predictions

In [ ]:
PROJECT_ROOT = Path("../").resolve()
PREDICTIONS_PATH = PROJECT_ROOT / "models/experiment_B/test_predictions.csv"

df = pd.read_csv(PREDICTIONS_PATH)
print(f"Loaded {len(df)} predictions.")

# Calculate baseline metrics
def calculate_metrics(df, pred_col="prediction", gt_col="ground_truth"):
    total = len(df)
    exact_match = (df[pred_col] == df[gt_col]).sum()
    
    try:
        c = cer(df[gt_col].tolist(), df[pred_col].tolist())
        w = wer(df[gt_col].tolist(), df[pred_col].tolist())
    except:
        c, w = 0, 0
        
    return {
        "CER": c * 100,
        "WER": w * 100,
        "Exact Match": (exact_match / total) * 100
    }

metrics_baseline = calculate_metrics(df)
print("Baseline Metrics (Experiment B):")
for k, v in metrics_baseline.items():
    print(f"  {k}: {v:.2f}%")

## 2. Phương pháp 1: SymSpell (Từ điển + Edit Distance)
Khảo sát các lỗi dạng ký tự như q/g, m/u và sửa dựa trên từ điển tần suất.
Để thực hiện, ta sẽ build từ điển từ tập Train Ground Truth.

In [ ]:
from symspellpy import SymSpell, Verbosity
from pyvi import ViTokenizer

# 1. Build dictionary from Train Annotations
TRAIN_GT_PATH = PROJECT_ROOT / "data/processed/train_annotation.txt"

sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)

print("Building dictionary...")
with open(TRAIN_GT_PATH, "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Processing lines"):
        parts = line.strip().split("\t")
        if len(parts) >= 2:
            text = parts[1]
            # Use simple tokenization (split by space) to build term frequencies
            for word in text.split():
                # Remove basic punctuation from words
                word_clean = re.sub(r"^\W+|\W+$", "", word)
                if word_clean:
                    sym_spell.create_dictionary_entry(word_clean, 1)

print(f"Dictionary built with {len(sym_spell.words)} unique words.")

In [ ]:
def symspell_correct(text):
    # Gương mặt cũ của SymSpell là không phân biệt uppercase/lowercase khi lookup trực tiếp, 
    # ta sẽ sửa theo từng từ
    words = text.split()
    corrected_words = []
    for word in words:
        # Extract punctuation
        prefix = ""
        suffix = ""
        core_word = word
        
        mo = re.match(r"^(\W*)(.*?)(\W*)$", word)
        if mo:
            prefix, core_word, suffix = mo.groups()
            
        if core_word == "":
            corrected_words.append(word)
            continue
            
        # Lookup
        suggestions = sym_spell.lookup(core_word, Verbosity.CLOSEST, max_edit_distance=1)
        if suggestions:
            best_match = suggestions[0].term
            # Restore capitalization if needed (simplistic logic)
            if core_word.istitle():
                best_match = best_match.title()
            elif core_word.isupper():
                best_match = best_match.upper()
            corrected_words.append(prefix + best_match + suffix)
        else:
            corrected_words.append(word)
    return " ".join(corrected_words)

# Test on a small sample first
print("Original:  guyên góp xây dựng")
print("Corrected:", symspell_correct("guyên góp xây dựng"))


In [ ]:
print("Applying SymSpell correction on test predictions...")
tqdm.pandas(desc="SymSpell")
df["prediction_symspell"] = df["prediction"].progress_apply(symspell_correct)

metrics_symspell = calculate_metrics(df, pred_col="prediction_symspell")
print("\nSymSpell Metrics:")
for k, v in metrics_symspell.items():
    print(f"  {k}: {v:.2f}%")

## 3. Phương pháp 2: Sử dụng BARTpho (Sequence-to-Sequence)
BARTpho là mô hình seq2seq chuyên cho tiếng Việt. Cách tốt nhất là coi bài toán sửa lỗi OCR như bài toán Translation: Đầu vào là text lỗi (từ OCR), đầu ra là text chuẩn. Tuy nhiên vì không huấn luyện (zero-shot/few-shot), ta có thể sử dụng các model được fine-tune sẵn cho Vietnamese Spelling Correction, hoặc dùng Text Generation API.
Ở đây, ta mô phỏng việc sửa lỗi với pretrained model pipeline. (Ví dụ: `bmd1905/vietnamese-correction-v2` dựa trên BARTpho).

In [ ]:
from transformers import pipeline

# Model: bmd1905/vietnamese-correction-v2 is a popular BartPho based spell-checker
device = 0 if torch.cuda.is_available() else -1

try:
    corrector = pipeline("text2text-generation", model="bmd1905/vietnamese-correction-v2", device=device)
    model_loaded = True
    print("Loaded BARTpho correction model.")
except Exception as e:
    print("Could not load correction model:", e)
    model_loaded = False

In [ ]:
if model_loaded:
    def bartpho_correct(text):
        # The model might need specific prefix or just raw text
        try:
            out = corrector(text, max_length=256)
            return out[0][generated_text]
        except:
            return text

    print("Original:  guyên góp xây dựng")
    print("Corrected:", bartpho_correct("guyên góp xây dựng"))
    
    # Apply to a sample if full takes too long on CPU, but since test set is 1637, it might take ~10 mins on GPU.
    # Un-comment the execution below when running in an environment with GPU
    """
    print("Applying BARTpho correction on test predictions...")
    tqdm.pandas(desc="BARTpho")
    df["prediction_bartpho"] = df["prediction"].progress_apply(bartpho_correct)

    metrics_bartpho = calculate_metrics(df, pred_col="prediction_bartpho")
    print("\nBARTpho Metrics:")
    for k, v in metrics_bartpho.items():
        print(f"  {k}: {v:.2f}%")
    """
else:
    print("Skipping evaluation as model loading failed.")

## 4. Kết luận & Phân tích

- Quan sát sự khác biệt metrics giữa Baseline, SymSpell và BARTpho.
- Trực quan hoá các dòng có thay đổi (trước/sau post-processing) để đánh giá chất lượng.

In [ ]:
if "prediction_symspell" in df.columns:
    # Show examples where symspell changed something and it resulted in exact match
    df["symspell_changed"] = df["prediction"] != df["prediction_symspell"]
    df["symspell_correct_now"] = df["prediction_symspell"] == df["ground_truth"]
    
    improved = df[df["symspell_changed"] & df["symspell_correct_now"]]
    print(f"SymSpell fixed {len(improved)} sentences perfectly.")
    
    if len(improved) > 0:
        print("\nExamples of fixes:")
        for idx, row in improved.head(5).iterrows():
            print(f"GT:    {row[ground_truth]}")
            print(f"Pred:  {row[prediction]}")
            print(f"Fixed: {row[prediction_symspell]}\n")